In [57]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch_geometric.datasets import KarateClub


In [39]:
import copy
graph = KarateClub()
data = copy.deepcopy(graph.data)
deg = torch.bincount(data.edge_index[0], minlength=34)
print(deg)
deg = torch.bincount(data.edge_index[1], minlength=34)
print(deg)

print(data.x.shape)
for i in data.x:
  print(i)

tensor([16,  9, 10,  6,  3,  4,  4,  4,  5,  2,  3,  1,  2,  5,  2,  2,  2,  2,
         2,  3,  2,  2,  2,  5,  3,  3,  2,  4,  3,  4,  4,  6, 12, 17])
tensor([16,  9, 10,  6,  3,  4,  4,  4,  5,  2,  3,  1,  2,  5,  2,  2,  2,  2,
         2,  3,  2,  2,  2,  5,  3,  3,  2,  4,  3,  4,  4,  6, 12, 17])
torch.Size([34, 34])
tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
tensor([0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
tensor([0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
tensor([0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
tensor([0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0

/tmp/ipykernel_1681238/1942779400.py:3: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  data = copy.deepcopy(graph.data)


In [45]:
print(data.x[0])
out = aggr_sum(data.x,data.edge_index)
print(out[0])


tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
tensor([0., 1., 1., 1., 1., 1., 1., 1., 1., 0., 1., 1., 1., 1., 0., 0., 0., 1.,
        0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.])


In [47]:

deg = torch.zeros(data.x.size(0))
deg.index_add_(0,data.edge_index[1], torch.ones(data.edge_index.size(1),))
print(deg)

out_mean = out/deg
print(out_mean[0])

out_mean = out/deg.clamp(min=1).unsqueeze(-1)
print(out_mean[0])
# deg.clamp(min=10).unsqueeze(-1)

tensor([16.,  9., 10.,  6.,  3.,  4.,  4.,  4.,  5.,  2.,  3.,  1.,  2.,  5.,
         2.,  2.,  2.,  2.,  2.,  3.,  2.,  2.,  2.,  5.,  3.,  3.,  2.,  4.,
         3.,  4.,  4.,  6., 12., 17.])
tensor([0.0000, 0.1111, 0.1000, 0.1667, 0.3333, 0.2500, 0.2500, 0.2500, 0.2000,
        0.0000, 0.3333, 1.0000, 0.5000, 0.2000, 0.0000, 0.0000, 0.0000, 0.5000,
        0.0000, 0.3333, 0.0000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.1667, 0.0000, 0.0000])
tensor([0.0000, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625,
        0.0000, 0.0625, 0.0625, 0.0625, 0.0625, 0.0000, 0.0000, 0.0000, 0.0625,
        0.0000, 0.0625, 0.0000, 0.0625, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0625, 0.0000, 0.0000])


In [ ]:

def aggr_sum(h : torch.Tensor, edge_index : torch.Tensor):
  out = torch.zeros(h.size(0), h.size(1), device=h.device, dtype=h.dtype)
  out.index_add_(0, edge_index[1], h[edge_index[0]])
  return out

def aggr_mean(h : torch.Tensor, edge_index : torch.Tensor):
  out = aggr_sum(h, edge_index)
  deg = torch.zeros(h.size(0),device=h.device)
  deg.index_add_(0,edge_index[1] ,torch.ones(edge_index.size(1),device=out.device))
  out = out / deg.clamp(min=1).unsqueeze(-1)
  return out

def get_aggr(aggr):
  if aggr == "sum":
        return aggr_sum
  elif aggr == "mean":
    return aggr_mean
  
  raise NotImplementedError()

class SageLayer(torch.nn.Module):
  def __init__(self, in_feat, out_feat, /, aggr="mean", normalize=True, device=None, ):
    super().__init__()
    self.normalize = normalize
    self.aggr_func = get_aggr(aggr)
    self.aggr = aggr
    self.lin_root = torch.nn.Linear(in_features=in_feat,out_features=out_feat,bias=True, device=device)
    self.lin_neigh = torch.nn.Linear(in_features=in_feat,out_features=out_feat,bias=False, device=device)


  def forward(self, x, edge_index):
    
    neigh_aggr = self.aggr_func(x, edge_index)
    
    h = self.lin_root(x) + self.lin_neigh(neigh_aggr)
    
    h = F.relu(h)

    if self.normalize:
      h = F.normalize(h,p=2,dim=-1)

    return h


In [ ]:
import torch
import torch.nn.functional as F

# --- aggr_mean corrigida ---
def aggr_sum(h, edge_index):
    out = torch.zeros(h.size(0), h.size(1), device=h.device, dtype=h.dtype)
    if edge_index.numel() > 0:
        out.index_add_(0, edge_index[1], h[edge_index[0]])
    return out

def aggr_mean(h, edge_index):
    out = aggr_sum(h, edge_index)
    if edge_index.numel() == 0:
        return out
    deg = torch.bincount(edge_index[1], minlength=h.size(0)).to(h.dtype)
    return out / deg.clamp(min=1).unsqueeze(-1)

# --- teste 1: single graph vs batch de 2 cópias ---
torch.manual_seed(0)
N, d = 4, 3
h_single = torch.randn(N, d)
ei_single = torch.tensor([[0, 1, 2, 3], [1, 2, 3, 0]])  # ciclo

out_single = aggr_mean(h_single, ei_single)

# Batch de 2 cópias, block-diagonal com offset
h_batch = torch.cat([h_single, h_single], dim=0)  # (8, 3)
ei_batch = torch.cat([ei_single, ei_single + N], dim=1)  # (2, 8)
out_batch = aggr_mean(h_batch, ei_batch)

assert torch.allclose(out_batch[:N], out_single), "Grafo 1 diverge"
assert torch.allclose(out_batch[N:], out_single), "Grafo 2 diverge"
print("✓ Batch block-diagonal preserva agregação por grafo")

# --- teste 2: bug antigo (torch.ones(N)) explodiria em E > N ---
# Aqui E = 4, N = 4 → E == N, então não pega. Force E > N:
ei_big = torch.tensor([[0, 1, 2, 3, 0, 1], [1, 2, 3, 0, 2, 3]])  # E=6 > N=4
out_ok = aggr_mean(h_single, ei_big)  # não crasha com bincount
print("✓ aggr_mean funciona com E > N")

In [ ]:
X = torch.tensor([
  [1, 0, 1],
  [0, 1, 0],
  [1, 0, 1]
], dtype=torch.float32)

W = torch.nn.Linear(3,5, dtype=torch.float32)
print(W.)
W(X)

tensor([[ 0.0248, -0.5383,  0.1465,  0.0194,  0.1435],
        [-0.2784, -0.0875, -0.0515,  0.0566, -0.2488],
        [-0.3132,  0.5139, -0.1732, -0.5692, -0.5419]],
       grad_fn=<PermuteBackward0>)


tensor([[-0.4433,  0.0549, -0.1914, -1.0796, -0.4633],
        [-0.4333, -0.0080, -0.2162, -0.4732, -0.3138],
        [-0.4433,  0.0549, -0.1914, -1.0796, -0.4633]],
       grad_fn=<AddmmBackward0>)